In [4]:
# SMARTCROP - SYNTHETIC SRI LANKAN CROP SUPPLY DATASET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

np.random.seed(42)
N = 15000

districts = [
    "Colombo","Gampaha","Kalutara","Kandy","Matale","Nuwara Eliya",
    "Galle","Matara","Hambantota","Jaffna","Kilinochchi","Mannar",
    "Mullaitivu","Vavuniya","Batticaloa","Ampara","Trincomalee",
    "Kurunegala","Puttalam","Anuradhapura","Polonnaruwa","Badulla",
    "Monaragala","Ratnapura","Kegalle"
]

# Sri Lankan-relevant crops (>30 classes)
crops = [
    "Carrot","Leeks","Beans","Chilli","Banana","Watermelon","Tomato",
    "Brinjal","Cabbage","Cauliflower","Pumpkin","Cucumber","Bitter Gourd",
    "Snake Gourd","Ridge Gourd","Okra","Red Onion","Big Onion","Potato",
    "Sweet Potato","Cassava","Maize","Rice","Groundnut","Green Gram",
    "Cowpea","Black Gram","Sesame","Soybean","Pineapple","Papaya","Mango",
    "Coconut","Tea","Pepper","Cinnamon","Ginger","Turmeric","Taro","Yam",
    "Long Bean","Ash Plantain"
]

seasons = ["Yala", "Maha"]

# Approximate production/area relationships (synthetic generation parameters).
# These are not official statistics.
crop_params = {
    "Carrot": (90, 18, 0.85), "Leeks": (75, 15, 0.80), "Beans": (65, 13, 0.75),
    "Chilli": (55, 11, 0.70), "Banana": (180, 35, 0.90), "Watermelon": (120, 25, 0.80),
    "Tomato": (85, 17, 0.78), "Brinjal": (80, 16, 0.76), "Cabbage": (95, 19, 0.82),
    "Cauliflower": (80, 16, 0.78), "Pumpkin": (100, 21, 0.74), "Cucumber": (95, 20, 0.76),
    "Bitter Gourd": (70, 14, 0.72), "Snake Gourd": (70, 14, 0.72),
    "Ridge Gourd": (68, 14, 0.72), "Okra": (60, 12, 0.70), "Red Onion": (65, 13, 0.75),
    "Big Onion": (70, 14, 0.78), "Potato": (110, 22, 0.85), "Sweet Potato": (90, 18, 0.72),
    "Cassava": (150, 30, 0.78), "Maize": (100, 20, 0.80), "Rice": (120, 24, 0.88),
    "Groundnut": (70, 14, 0.70), "Green Gram": (55, 11, 0.68), "Cowpea": (55, 11, 0.68),
    "Black Gram": (50, 10, 0.66), "Sesame": (45, 9, 0.64), "Soybean": (65, 13, 0.72),
    "Pineapple": (130, 27, 0.85), "Papaya": (150, 30, 0.86), "Mango": (140, 28, 0.84),
    "Coconut": (170, 34, 0.90), "Tea": (140, 28, 0.88), "Pepper": (40, 8, 0.65),
    "Cinnamon": (35, 7, 0.62), "Ginger": (75, 15, 0.76), "Turmeric": (70, 14, 0.74),
    "Taro": (85, 17, 0.72), "Yam": (90, 18, 0.73), "Long Bean": (65, 13, 0.70),
    "Ash Plantain": (175, 35, 0.88)
}

# Crop weights: give more observations to your main focus crops
focus = {"Carrot","Leeks","Beans","Chilli","Banana","Watermelon"}
weights = np.array([4 if c in focus else 1 for c in crops], dtype=float)
weights /= weights.sum()

# District-level relative agricultural scale (synthetic)
district_scale = {
    d: np.random.uniform(0.75, 1.35) for d in districts
}

# Crop x district suitability/production multipliers
# Keeps values varied rather than assigning identical supply to every district.
zone_preference = {
    "Carrot": {"Nuwara Eliya":1.8,"Badulla":1.4,"Kandy":1.2,"Matale":1.1},
    "Leeks": {"Nuwara Eliya":1.9,"Badulla":1.4,"Kandy":1.2},
    "Beans": {"Nuwara Eliya":1.5,"Badulla":1.4,"Kandy":1.2},
    "Chilli": {"Anuradhapura":1.5,"Monaragala":1.4,"Hambantota":1.3,"Puttalam":1.2},
    "Banana": {"Kurunegala":1.4,"Monaragala":1.4,"Anuradhapura":1.3,"Hambantota":1.2},
    "Watermelon": {"Hambantota":1.8,"Monaragala":1.5,"Anuradhapura":1.4,"Puttalam":1.3},
    "Big Onion": {"Matale":1.5,"Anuradhapura":1.3,"Polonnaruwa":1.2},
    "Red Onion": {"Jaffna":1.8,"Kilinochchi":1.5,"Mannar":1.3},
    "Rice": {"Anuradhapura":1.5,"Polonnaruwa":1.7,"Ampara":1.6,"Kurunegala":1.3},
    "Tea": {"Nuwara Eliya":2.0,"Kandy":1.5,"Badulla":1.5,"Ratnapura":1.3},
    "Coconut": {"Kurunegala":1.7,"Puttalam":1.5,"Gampaha":1.3},
    "Cinnamon": {"Galle":1.8,"Matara":1.6,"Kalutara":1.4}
}

records = []
for i in range(N):
    district = np.random.choice(districts, p=None)
    season = np.random.choice(seasons, p=[0.45, 0.55])
    crop = np.random.choice(crops, p=weights)
    year = np.random.randint(2018, 2027)

    base_area, base_yield, efficiency = crop_params[crop]
    multiplier = district_scale[district]
    multiplier *= zone_preference.get(crop, {}).get(district, 1.0)

    # Seasonal effect: some crops get different cultivated area/supply in Yala vs Maha
    seasonal_factor = np.random.uniform(0.85, 1.15)
    if season == "Maha":
        seasonal_factor *= np.random.uniform(1.00, 1.12)

    # Gradual year-to-year change, plus random local variation
    year_factor = 1 + (year - 2018) * np.random.uniform(-0.005, 0.015)

    cultivated_area = (
        base_area * multiplier * seasonal_factor * year_factor *
        np.random.lognormal(mean=0, sigma=0.22)
    )
    cultivated_area = max(cultivated_area, 1.0)

    # Production = area × synthetic yield, with realistic random variation
    yield_per_area = base_yield * efficiency * np.random.lognormal(mean=0, sigma=0.12)
    production = cultivated_area * yield_per_area

    records.append({
        "District": district,
        "Season": season,
        "Crop": crop,
        "Year": year,
        "Cultivated Area": round(cultivated_area, 2),
        "Production": round(production, 2)
    })



# Save exactly the requested six-feature dataset
output_file = "sri_lanka_crop_supply_data.csv"
df[["District","Season","Crop","Year","Cultivated Area","Production"]].to_csv(
    output_file, index=False
)

print("\nCSV CREATED:", output_file)
files.download(output_file)



CSV CREATED: sri_lanka_crop_supply_data.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>